In [3]:
import re, json
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
import litellm

/Users/monusingh/work-share/code-blogs-articles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
sample_doc = """
Machine learning is a subset of artificial intelligence that enables systems to learn from data.
It uses statistical techniques to give computers the ability to improve with experience.
Supervised learning requires labeled training data to make predictions.
The model learns a mapping from inputs to outputs based on example pairs.

Neural networks are computing systems inspired by biological neural networks in animal brains.
They consist of layers of interconnected nodes that process information using connectionist approaches.
Deep learning uses many layers to learn hierarchical representations of data.
Convolutional networks are especially effective for image recognition tasks.

Natural language processing allows machines to understand and generate human language.
Transformers have revolutionized NLP by using self-attention mechanisms.
BERT and GPT are prominent examples of transformer-based language models.
These models are pretrained on massive corpora and then fine-tuned for specific tasks.
"""

In [ ]:
def character_chunk(text: str, size: int = 200, overlap: int = 20) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]

def word_chunk(text: str, size: int = 50, overlap: int = 10) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        chunks.append(" ".join(words[i : i + size]))
    return [c for c in chunks if c.strip()]

def sentence_chunk(text: str, sentences_per_chunk: int = 2) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunks.append(" ".join(sentences[i : i + sentences_per_chunk]))
    return chunks

def paragraph_chunk(text: str) -> List[str]:
    # splits on blank lines
    paragraphs = re.split(r'\n\s*\n', text.strip())
    return [p.strip() for p in paragraphs if p.strip()]

def recursive_chunk(text: str, size: int = 200, overlap: int = 20,
                    separators: List[str] = ["\n\n", "\n", ". ", " ", ""]) -> List[str]:
    for sep in separators:
        parts = text.split(sep) if sep else list(text)
        parts = [p for p in parts if p.strip()]
        # if splitting produces reasonably sized pieces, use it
        if all(len(p) <= size * 1.5 for p in parts):
            # merge small adjacent parts up to `size`
            chunks, current = [], ""
            for part in parts:
                candidate = (current + sep + part).strip() if current else part
                if len(candidate) <= size:
                    current = candidate
                else:
                    if current:
                        chunks.append(current)
                    current = part
            if current:
                chunks.append(current)
            return chunks
    return [text]

model = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_chunk(text: str, threshold: float = 0.5) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    embeddings = model.encode(sentences)

    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
        if sim < threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentences[i])

    if current:
        chunks.append(" ".join(current))
    return chunks

In [9]:
strategies = {
    "Character (200, ov=20)":  character_chunk(sample_doc, size=200, overlap=20),
    "Word (50w, ov=10)":       word_chunk(sample_doc, size=50, overlap=10),
    "Sentence (2 per chunk)":  sentence_chunk(sample_doc, sentences_per_chunk=2),
    "Paragraph":               paragraph_chunk(sample_doc),
    "Recursive (200)":         recursive_chunk(sample_doc, size=200),
    "Semantic (t=0.5)":        semantic_chunk(sample_doc, threshold=0.5),
}

rows = []
for name, chunks in strategies.items():
    lengths = [len(c) for c in chunks]
    rows.append({
        "Strategy":    name,
        "Num Chunks":  len(chunks),
        "Min Chars":   min(lengths),
        "Max Chars":   max(lengths),
        "Avg Chars":   round(np.mean(lengths)),
        "Std Dev":     round(np.std(lengths)),
    })

df = pd.DataFrame(rows).set_index("Strategy")
print(df.to_string())

                        Num Chunks  Min Chars  Max Chars  Avg Chars  Std Dev
Strategy                                                                    
Character (200, ov=20)           6        109        200        185       34
Word (50w, ov=10)                4        133        395        305      102
Sentence (2 per chunk)           6        145        198        167       18
Paragraph                        3        320        353        335       14
Recursive (200)                  6        145        198        167       18
Semantic (t=0.5)                 9         71        233        111       49


In [16]:
from langchain_text_splitters import (          # ← new package
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Character
char_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)

# 2. Recursive (default in most LangChain RAG pipelines)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 3. Token-aware (respects model token limits)
token_splitter = SentenceTransformersTokenTextSplitter(
    model_name="all-MiniLM-L6-v2",
    chunk_overlap=10
)

# 4. Semantic (uses embeddings, equivalent to what we built manually)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",   # or "standard_deviation", "interquartile"
    breakpoint_threshold_amount=85
)

chunks = semantic_splitter.split_text(sample_doc)

/var/folders/0l/p7lzlqxn44b036b_3ykx4tlr0000gn/T/ipykernel_6552/1295058332.py:6: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1495.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/0l/p7lzlqxn44b036b_3ykx4tlr0000gn/T/ipykernel_6552/1295058332.py:26: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updat

In [18]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
    TokenTextSplitter,
    MarkdownNodeParser,
    CodeSplitter,                  # language-aware splitting for code
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

# Sentence-aware splitter (most common default in LlamaIndex)
sentence_parser = SentenceSplitter(chunk_size=256, chunk_overlap=20)

# Semantic splitter
semantic_parser = SemanticSplitterNodeParser(
    buffer_size=1,                 # sentences to look ahead/behind
    breakpoint_percentile_threshold=95,
    embed_model=embed_model
)

from llama_index.core import Document
doc = Document(text=sample_doc)
nodes = semantic_parser.get_nodes_from_documents([doc])
chunks = [n.get_content() for n in nodes]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1643.32it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
chunks

['\nMachine learning is a subset of artificial intelligence that enables systems to learn from data.\nIt uses statistical techniques to give computers the ability to improve with experience.\nSupervised learning requires labeled training data to make predictions.\nThe model learns a mapping from inputs to outputs based on example pairs.\n\nNeural networks are computing systems inspired by biological neural networks in animal brains.\nThey consist of layers of interconnected nodes that process information using connectionist approaches.\nDeep learning uses many layers to learn hierarchical representations of data.\nConvolutional networks are especially effective for image recognition tasks.\n\nNatural language processing allows machines to understand and generate human language.\n',
 'Transformers have revolutionized NLP by using self-attention mechanisms.\nBERT and GPT are prominent examples of transformer-based language models.\nThese models are pretrained on massive corpora and then 

#### Semantic chunking via similarity threshold

A manual implementation: walk sentences, start a new chunk whenever cosine similarity to the previous sentence drops below a threshold.


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "LLMs have limited context windows.",
    "This makes naive prompt stuffing impractical.",
    "So we rely on retrieval-augmented generation (RAG) to bring only relevant context.",
    "Chunking is a core retrieval strategy in RAG pipelines.",
    "Chunks are embedded into vectors that capture semantic meaning.",
    "Vector databases store embeddings efficiently and support fast lookup.",
    "At query time, approximate nearest neighbor (ANN) search retrieves candidate chunks.",
    "A reranker can reorder retrieved chunks for higher precision.",
    "Monitoring is critical once models are deployed.",
    "You track latency, quality drift, and retrieval hit-rates to catch regressions early.",
]

embeddings = model.encode(sentences)

SIM_THRESHOLD = 0.55

chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        embeddings[i-1].reshape(1, -1),
        embeddings[i].reshape(1, -1)
    )[0][0]

    if sim < SIM_THRESHOLD:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
    else:
        current_chunk.append(sentences[i])

chunks.append(" ".join(current_chunk))

for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---\n{c}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Chunk 1 ---
LLMs have limited context windows.

--- Chunk 2 ---
This makes naive prompt stuffing impractical.

--- Chunk 3 ---
So we rely on retrieval-augmented generation (RAG) to bring only relevant context. Chunking is a core retrieval strategy in RAG pipelines.

--- Chunk 4 ---
Chunks are embedded into vectors that capture semantic meaning.

--- Chunk 5 ---
Vector databases store embeddings efficiently and support fast lookup.

--- Chunk 6 ---
At query time, approximate nearest neighbor (ANN) search retrieves candidate chunks. A reranker can reorder retrieved chunks for higher precision.

--- Chunk 7 ---
Monitoring is critical once models are deployed.

--- Chunk 8 ---
You track latency, quality drift, and retrieval hit-rates to catch regressions early.


#### AST-based code chunking

For code, splitting on characters/tokens risks cutting a function or class in half. Parse to an AST and chunk on `ClassDef`/`FunctionDef` boundaries instead.


In [6]:
import ast

code = """
class CreditScorer:
    def __init__(self, model):
        self.model = model

    def preprocess(self, data):
        return data.fillna(0)

    def score(self, data):
        data = self.preprocess(data)
        return self.model.predict(data)

def helper(x):
    return x * 2
"""

tree = ast.parse(code)

chunks = []

for node in tree.body:
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        chunk = ast.get_source_segment(code, node)
        chunks.append(chunk)

for i, c in enumerate(chunks):
    print(f"\n--- Code Chunk {i+1} ---\n{c}")


--- Code Chunk 1 ---
class CreditScorer:
    def __init__(self, model):
        self.model = model

    def preprocess(self, data):
        return data.fillna(0)

    def score(self, data):
        data = self.preprocess(data)
        return self.model.predict(data)

--- Code Chunk 2 ---
def helper(x):
    return x * 2
